# 7.5 功耗管理与电池感知调度

## 目的

端侧电池是硬约束。需要把 **功耗建模 → DVFS → 电量档位调度 → 精度/模型降级** 串成可落地策略。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

import math

## 7.5.1 功耗建模

$$P = \alpha C V^2 f + V I_{leak}$$

LLM 特征：Prefill 偏计算密集（高功耗），Decode 偏带宽受限（功耗相对较低），NPU 能效通常高于 CPU/GPU。

In [ ]:
def dynamic_power(alpha, C, V, f):
    return alpha * C * (V ** 2) * f


def total_power(alpha, C, V, f, I_leak):
    return dynamic_power(alpha, C, V, f) + V * I_leak


# 简化 SoC 三档频率
profiles = {
    "high":  dict(V=0.95, f=1.0e9, alpha=0.7),
    "mid":   dict(V=0.85, f=0.7e9, alpha=0.55),
    "low":   dict(V=0.75, f=0.4e9, alpha=0.4),
}
C, I_leak = 2.5e-9, 0.15  # 示意参数
print("=== 功耗档位对比（示意）===")
for name, p in profiles.items():
    P = total_power(p["alpha"], C, p["V"], p["f"], I_leak)
    print(f"{name:4s}: V={p['V']:.2f}V  f={p['f']/1e9:.1f}GHz  P≈{P*1000:.1f} mW (相对)")

## 7.5.2 DVFS：在延迟预算下选最低功耗档

In [ ]:
def estimate_latency_ms(tokens, tok_per_s):
    return tokens / tok_per_s * 1000


# 各档吞吐（示意）
throughput = {"high": 35, "mid": 24, "low": 14}
deadline_ms = 1200  # 生成 24 token 的延迟预算
n_tokens = 24

print("=== DVFS 调度 ===")
chosen = None
for name in ("low", "mid", "high"):
    lat = estimate_latency_ms(n_tokens, throughput[name])
    ok = lat <= deadline_ms
    print(f"{name}: {throughput[name]} tok/s → {lat:.0f}ms  {'OK' if ok else 'NO'}")
    if ok and chosen is None:
        chosen = name
print(f"选择最低满足档位: {chosen}")

## 7.5.3 电池感知策略 + 精度自适应

In [ ]:
def battery_policy(soc_pct: float) -> dict:
    if soc_pct > 50:
        return {"model": "3B", "quant": "W4A16", "dvfs": "high", "cloud_fallback": False}
    if soc_pct > 20:
        return {"model": "1.5B", "quant": "W4A16", "dvfs": "mid", "cloud_fallback": False}
    if soc_pct > 10:
        return {"model": "1.5B", "quant": "W4A8", "dvfs": "low", "cloud_fallback": False}
    return {"model": "cloud", "quant": "-", "dvfs": "off", "cloud_fallback": True}


def thermal_policy(temp_c: float, base: dict) -> dict:
    out = dict(base)
    if temp_c >= 45:
        out["dvfs"] = "low"
        out["quant"] = "W4A8"
    if temp_c >= 50:
        out["model"] = "1.5B"
        out["cloud_fallback"] = True
    return out


print("=== 电量 × 温度联合策略 ===")
for soc in (80, 35, 15, 5):
    for temp in (38, 48):
        pol = thermal_policy(temp, battery_policy(soc))
        print(f"SoC={soc:2d}% T={temp}°C → {pol}")

## 7.5.4 续航估算：一轮对话耗电

In [ ]:
def estimate_mah(prefill_ms, decode_ms, p_prefill_w, p_decode_w, battery_mah=5000, vbat=3.8):
    """粗算：能量(J)=P*t，再换算 mAh。"""
    e_j = p_prefill_w * (prefill_ms/1000) + p_decode_w * (decode_ms/1000)
    mah = e_j / (vbat * 3.6) * 1000 / 1000  # J / (V*3.6) = mAh
    # 上面简化：1 mWh = 3.6 J；mAh = Wh/V *1000
    mah = e_j / (vbat * 3.6)
    turns = battery_mah * 0.2 / mah  # 假设 AI 可用 20% 电量
    return mah, turns


mah, turns = estimate_mah(80, 900, p_prefill_w=3.5, p_decode_w=1.8)
print("=== 单轮对话续航粗算 ===")
print(f"单轮约耗电: {mah:.3f} mAh")
print(f"20% 电量可支撑约: {turns:.0f} 轮对话（示意，实测请用 5.4 热节流基准）")

## 小结

1. Prefill/Decode 功耗画像不同，调度应分阶段。
2. DVFS：在延迟 SLA 下选最低档。
3. 电量低 / 过热：先降量化与模型尺寸，再云端回退。
4. 与第 5.4 节硬件基准测试配合，用真机校准参数。